In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity


print("Loading dataset...")
df = pd.read_csv("../data/cleaned_arxiv_papers.csv")

print("Loading embeddings...")
embeddings = np.load("../data/arxiv_embeddings.npy")


print("Loading model...")
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

print("System Ready! Embeddings shape:", embeddings.shape)

Loading dataset...
Loading embeddings...
Loading model...
System Ready! Embeddings shape: (50000, 384)


In [2]:
print(embeddings.shape)
print(embeddings.dtype)
embeddings

(50000, 384)
float32


array([[-0.13156408, -0.00678264, -0.00367609, ...,  0.09561247,
        -0.04423941,  0.00522624],
       [-0.03589669, -0.06343268, -0.00394966, ...,  0.07206137,
        -0.13262013, -0.0041208 ],
       [-0.08129285,  0.02410299, -0.08868261, ..., -0.04524419,
        -0.0476585 , -0.01549935],
       ...,
       [-0.00546027,  0.00445373,  0.10184892, ..., -0.0455995 ,
        -0.04973711, -0.01499986],
       [ 0.03493429, -0.00508128,  0.09953675, ..., -0.01594462,
        -0.05550025, -0.03442113],
       [-0.03093104, -0.02449019, -0.00522512, ..., -0.00051519,
        -0.04664399,  0.00573538]], shape=(50000, 384), dtype=float32)

In [3]:
import faiss

# 1. We must make a copy of the embeddings first because FAISS normalizes "in-place"
# (Meaning it permanently alters the original variable)
faiss_embeddings = embeddings.copy()

# 2. Apply L2 Normalization
faiss.normalize_L2(faiss_embeddings)

print("Embeddings normalized! Ready for the FAISS Index.")

Embeddings normalized! Ready for the FAISS Index.


In [4]:
# index = faiss.IndexFlatIP(384)
# index.add(faiss_embeddings)
# index

In [5]:
import os

# Define the exact path to your data folder
index_path = "../data/paper_faiss.index"

if os.path.exists(index_path):
    print("Loading existing FAISS index")
    index = faiss.read_index(index_path)
else:
    print("Creating new FAISS index")
    faiss.normalize_L2(embeddings)
    index = faiss.IndexFlatIP(384)
    index.add(embeddings)
    
    # Save it using the new path variable
    faiss.write_index(index, index_path)
    print("FAISS index saved successfully to the data folder!")

Loading existing FAISS index


In [6]:
query = "deep learning for medical image analysis"
query_embedding = model.encode(query)
print(query_embedding.shape)

(384,)


In [7]:
query_embedding = model.encode([query])
# query_embedding = model.encode(query).reshape(1,-1)
print(query_embedding.shape)
print(query_embedding)

(1, 384)
[[-3.52526680e-02  5.43267019e-02  5.65811843e-02 -4.97096367e-02
  -3.52174528e-02 -2.67348811e-02 -8.23491812e-03  1.52610447e-02
  -1.31554995e-02 -5.45782186e-02 -5.26165217e-02 -1.20118242e-02
  -8.10011178e-02  9.73842740e-02 -1.15813106e-01 -1.68416556e-02
  -9.74306762e-02 -5.52446954e-03 -1.06863745e-01  7.64731225e-03
  -4.89180125e-02 -6.45616604e-03 -3.65177132e-02  2.23113857e-02
   5.65222204e-02  4.34990693e-03  6.29341602e-02 -1.09062947e-01
   3.48555110e-02 -1.03745852e-02  7.24759400e-02 -3.63332741e-02
   1.78154167e-02  1.52814882e-02  4.68112044e-02  7.27785975e-02
  -1.72747020e-02  5.71634509e-02  6.73017837e-03  2.91875172e-02
   4.84563373e-02  5.15047163e-02  3.13240029e-02  5.97998463e-02
   8.63463879e-02 -1.21602137e-03  3.83908004e-02 -1.21437116e-02
   5.59613705e-02  9.04097781e-02 -1.46837588e-02  2.19263658e-02
  -7.75724500e-02  8.67380500e-02  3.64489108e-02 -1.94056355e-03
  -1.39826890e-02 -1.21372230e-02 -1.06874995e-01 -9.59566329e-03
 

In [8]:
faiss.normalize_L2(query_embedding)
query_embedding

array([[-3.52526680e-02,  5.43267019e-02,  5.65811843e-02,
        -4.97096367e-02, -3.52174528e-02, -2.67348811e-02,
        -8.23491812e-03,  1.52610447e-02, -1.31554995e-02,
        -5.45782186e-02, -5.26165217e-02, -1.20118242e-02,
        -8.10011178e-02,  9.73842740e-02, -1.15813106e-01,
        -1.68416556e-02, -9.74306762e-02, -5.52446954e-03,
        -1.06863745e-01,  7.64731225e-03, -4.89180125e-02,
        -6.45616604e-03, -3.65177132e-02,  2.23113857e-02,
         5.65222204e-02,  4.34990693e-03,  6.29341602e-02,
        -1.09062947e-01,  3.48555110e-02, -1.03745852e-02,
         7.24759400e-02, -3.63332741e-02,  1.78154167e-02,
         1.52814882e-02,  4.68112044e-02,  7.27785975e-02,
        -1.72747020e-02,  5.71634509e-02,  6.73017837e-03,
         2.91875172e-02,  4.84563373e-02,  5.15047163e-02,
         3.13240029e-02,  5.97998463e-02,  8.63463879e-02,
        -1.21602137e-03,  3.83908004e-02, -1.21437116e-02,
         5.59613705e-02,  9.04097781e-02, -1.46837588e-0

In [9]:
D, I = index.search(query_embedding, 5)

print("Scores (D):", D)
print("Indices (I):", I)

Scores (D): [[0.72542244 0.71706575 0.7123304  0.68072456 0.67988193]]
Indices (I): [[26063 37161 18030 10466 29766]]


In [10]:
print(df.iloc[26063]['title'])

An overview of deep learning in medical imaging focusing on MRI


In [11]:
def search_paper(query,k=5):
    query_embedding=model.encode([query])
    faiss.normalize_L2(query_embedding)
    D,I=index.search(query_embedding,k)
    for score,idx in zip(D[0],I[0]):
        print("Similarity Score: ",score)
        print("Title: ",df.iloc[idx]['title'])
        print("Abstract:" ,df.iloc[idx]['abstract'][:500])
        print() # for space between two papers
    return D,I

In [13]:
import transformers
from transformers import pipeline
summarizer = pipeline(
    "summarization",
    model="sshleifer/distilbart-cnn-12-6",  # 300MB instead of 1.6GB
    device=0
)

In [14]:
type(summarizer)

transformers.pipelines.text2text_generation.SummarizationPipeline

In [26]:
def search_and_summarize(query, k=5):
    query_embedding = model.encode([query])
    faiss.normalize_L2(query_embedding)
    D, I = index.search(query_embedding, k)
    
    for score, idx in zip(D[0], I[0]):
        print("Similarity score:", score)
        print("Title:", df.iloc[idx]["title"])
        print("Abstract:", df.iloc[idx]["abstract"][:500])
        print()
        
        # This MUST be indented inside the loop to summarize every paper!
        summary = summarizer(df.iloc[idx]["abstract"], max_length=120, min_length=40, do_sample=False)
        print(summary)
        print("AI SUMMARY:", summary[0]["summary_text"])
        print("-" * 80) # Added a divider line to keep the output readable

In [24]:
from keybert import KeyBERT

# kw_model=keybert()
kw_model=KeyBERT(model=model) # as by defualt it may take some another model, so we want the same model to create embeeding the earlier one we used in sentence transofmrer we will use same model
type(kw_model)

keybert._model.KeyBERT

In [28]:
text=df.iloc[26063]['abstract']
keywords=kw_model.extract_keywords(text)
print(keywords)
print(type(keywords))
print(type(keywords[0]))

[('mri', 0.4733), ('neural', 0.3921), ('imaging', 0.3825), ('deep', 0.3559), ('networks', 0.2893)]
<class 'list'>
<class 'tuple'>


In [ ]:
finalkeyword=kw_model.extract_keywords(text, keyphrase_ngram_range=(1, 3), stop_words=None)
finalkeyword

[('learning in mri', 0.5728),
 ('how deep learning', 0.5452),
 ('deep neural', 0.5435),
 ('deep learning', 0.5202),
 ('on deep learning', 0.5181)]